# Seduh Coffee — Data Cleaning
**Proyek Akhir Data Analyst (BNSP) · Talentiv**

Notebook ini menggabungkan proses cleaning dari 4 sheet di file Excel sumber:

| Sheet | Baris awal → akhir | Yang dibersihkan |
|---|---|---|
| **Orders** | 25.804 → 25.776 | 14 baris duplikat dihapus, 14 baris `quantity` ≤ 0 dihapus, penulisan `channel` & `city` diseragamkan, tipe data diperbaiki |
| **Customers** | 15.000 → 15.000 | nama, nomor HP, usia, gender, kota/provinsi/channel diseragamkan; 380 kota kosong diisi `Unknown` |
| **Products** | 50 → 50 | `category` (9 variasi tulisan → 6 kategori), harga & cost jadi angka, `launch_date` jadi tanggal |
| **Marketing_Spend** | 96 → 96 | `month` jadi tanggal; dicek 24 bulan × 4 channel, tanpa duplikat |

Hasil akhir tiap sheet disimpan sebagai CSV di bagian **Export** (paling bawah).

**Daftar isi:** 1. Orders · 2. Customers · 3. Products · 4. Marketing_Spend · 5. Export

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import re

# Ganti path ini kalau file Excel sumbernya pindah lokasi (cukup di satu tempat ini)
FILE_PATH = r"C:\Users\THINKPAD\Documents\Talentiv\Day 8\Seduh Coffee - Proyek Akhir Data Analyst (BNSP).xlsx"

## 1. Orders

In [ ]:
df_seduh_order_raw = pd.read_excel(FILE_PATH, sheet_name='Orders', dtype=str)
df_seduh_order = df_seduh_order_raw.copy()

### 1.1 Cek struktur awal

In [1]:
df_seduh_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 25804 entries, 0 to 25803
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        25804 non-null  str  
 1   order_date      25804 non-null  str  
 2   customer_id     25804 non-null  str  
 3   product_id      25804 non-null  str  
 4   quantity        25804 non-null  str  
 5   unit_price      25804 non-null  str  
 6   discount_pct    25804 non-null  str  
 7   channel         25804 non-null  str  
 8   payment_method  25804 non-null  str  
 9   city            25804 non-null  str  
 10  province        25804 non-null  str  
 11  order_status    25804 non-null  str  
 12  rating          21723 non-null  str  
 13  review_text     9096 non-null   str  
dtypes: str(14)
memory usage: 2.8 MB


In [2]:
df_seduh_order.columns

Index(['order_id', 'order_date', 'customer_id', 'product_id', 'quantity',
       'unit_price', 'discount_pct', 'channel', 'payment_method', 'city',
       'province', 'order_status', 'rating', 'review_text'],
      dtype='str')

### 1.2 `order_id` — cek format & duplikat

In [3]:
df_seduh_order.loc[~df_seduh_order['order_id'].astype(str).str.fullmatch(r'ORD-\d{4}-\d{6}')]

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel,payment_method,city,province,order_status,rating,review_text


In [4]:
df_seduh_order['order_id'].duplicated().sum()

np.int64(14)

In [5]:
df_seduh_order.loc[df_seduh_order['order_id'].duplicated(keep=False)].sort_values('order_id')

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel,payment_method,city,province,order_status,rating,review_text
1181,ORD-2024-001182,2024-02-14 00:00:00,CUST-02508,PRD-033,1,89000,0,Shopee,Paylater,Malang,Jawa Timur,Completed,4,NaN
1182,ORD-2024-001182,2024-02-14 00:00:00,CUST-02508,PRD-033,1,89000,0,Shopee,Paylater,Malang,Jawa Timur,Completed,4,NaN
2278,ORD-2024-002278,2024-03-21 00:00:00,CUST-02592,PRD-042,1,289000,0.1,Shopee,COD,Depok,Jawa Barat,Completed,4,NaN
2279,ORD-2024-002278,2024-03-21 00:00:00,CUST-02592,PRD-042,1,289000,0.1,Shopee,COD,Depok,Jawa Barat,Completed,4,NaN
3023,ORD-2024-003022,2024-04-13 00:00:00,CUST-09706,PRD-008,1,195000,0,Shopee,COD,Tangerang,Banten,Completed,NaN,NaN
3024,ORD-2024-003022,2024-04-13 00:00:00,CUST-09706,PRD-008,1,195000,0,Shopee,COD,Tangerang,Banten,Completed,NaN,NaN
7508,ORD-2024-007506,2024-09-09 00:00:00,CUST-06097,PRD-020,4,165000,0,TikTok Shop,OVO,Surabaya,Jawa Timur,Completed,3,NaN
7509,ORD-2024-007506,2024-09-09 00:00:00,CUST-06097,PRD-020,4,165000,0,TikTok Shop,OVO,Surabaya,Jawa Timur,Completed,3,NaN
7533,ORD-2024-007530,10/09/2024,CUST-00681,PRD-011,2,135000,0.25,Webstore,DANA,Balikpapan,Kalimantan Timur,Completed,5,NaN
7534,ORD-2024-007530,10/09/2024,CUST-00681,PRD-011,2,135000,0.25,Webstore,DANA,Balikpapan,Kalimantan Timur,Completed,5,NaN


In [6]:
df_seduh_order = df_seduh_order.drop_duplicates().copy()
df_seduh_order.info()

<class 'pandas.DataFrame'>
Index: 25790 entries, 0 to 25803
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        25790 non-null  str  
 1   order_date      25790 non-null  str  
 2   customer_id     25790 non-null  str  
 3   product_id      25790 non-null  str  
 4   quantity        25790 non-null  str  
 5   unit_price      25790 non-null  str  
 6   discount_pct    25790 non-null  str  
 7   channel         25790 non-null  str  
 8   payment_method  25790 non-null  str  
 9   city            25790 non-null  str  
 10  province        25790 non-null  str  
 11  order_status    25790 non-null  str  
 12  rating          21710 non-null  str  
 13  review_text     9093 non-null   str  
dtypes: str(14)
memory usage: 3.0 MB


### 1.3 `order_date` → datetime

In [7]:
df_seduh_order['order_date']=pd.to_datetime(
            df_seduh_order['order_date']
            .astype(str)
            .str.strip()
            .str.replace(r'^(\d{2})/(\d{2})/(\d{4})', r'\3-\2-\1', regex=True),
            format='mixed', errors='coerce'
)

In [8]:
print(df_seduh_order['order_date'].isna().sum())
print(df_seduh_order['order_date'].agg(['min', 'max']))
print(df_seduh_order.sort_values('order_id')['order_date'].is_monotonic_increasing)
print(df_seduh_order.info())

0
min   2024-01-01
max   2025-12-31
Name: order_date, dtype: datetime64[us]
True
<class 'pandas.DataFrame'>
Index: 25790 entries, 0 to 25803
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        25790 non-null  str           
 1   order_date      25790 non-null  datetime64[us]
 2   customer_id     25790 non-null  str           
 3   product_id      25790 non-null  str           
 4   quantity        25790 non-null  str           
 5   unit_price      25790 non-null  str           
 6   discount_pct    25790 non-null  str           
 7   channel         25790 non-null  str           
 8   payment_method  25790 non-null  str           
 9   city            25790 non-null  str           
 10  province        25790 non-null  str           
 11  order_status    25790 non-null  str           
 12  rating          21710 non-null  str           
 13  review_text     9093 non-null   str      

### 1.4 `customer_id` & `product_id` — cek format

In [9]:
df_seduh_order.loc[~df_seduh_order['customer_id'].astype(str).str.fullmatch(r'CUST-\d{5}')]

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel,payment_method,city,province,order_status,rating,review_text


In [10]:
df_seduh_order.loc[~df_seduh_order['product_id'].astype(str).str.fullmatch(r'PRD-\d{3}')]

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel,payment_method,city,province,order_status,rating,review_text


### 1.5 `quantity` — ubah ke angka, buang baris ≤ 0

In [11]:
df_seduh_order.loc[
    ~df_seduh_order['quantity'].str.fullmatch(r'\d+').fillna(False),
    'quantity'
].value_counts(dropna=False)

quantity
-3    4
-1    3
-2    2
Name: count, dtype: int64

In [12]:
df_seduh_order['quantity'].value_counts()

quantity
1     8070
2     7234
3     4983
4     2925
5     1389
6      679
7      292
8      114
9       62
10      20
11       6
0        5
-3       4
-1       3
12       2
-2       2
Name: count, dtype: int64

In [13]:
df_seduh_order['quantity']=pd.to_numeric(
                    df_seduh_order['quantity']
                    .str.strip(),
                    errors='coerce'
)

In [14]:
print((df_seduh_order['quantity'] <= 0).sum())   # harusnya 14
df_seduh_order = df_seduh_order[df_seduh_order['quantity'] > 0].copy()

14


In [15]:
print(df_seduh_order['quantity'].isna().sum())
print(df_seduh_order['quantity'].describe())

0
count    25776.000000
mean         2.483318
std          1.506595
min          1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         12.000000
Name: quantity, dtype: float64


### 1.6 `unit_price` & `discount_pct` → numerik

In [16]:
df_seduh_order.loc[~df_seduh_order['unit_price'].str.fullmatch(r'\d+')]

,order_id,order_date,customer_id,product_id,quantity,unit_price,discount_pct,channel,payment_method,city,province,order_status,rating,review_text


In [17]:
df_seduh_order['unit_price']=pd.to_numeric(
                df_seduh_order['unit_price']
                .str.strip(),
                errors='coerce'
)

In [18]:
df_seduh_order['unit_price'].describe()
df_seduh_order['unit_price'].isna().sum()

np.int64(0)

In [19]:
df_seduh_order['discount_pct'].value_counts(dropna=False)

discount_pct
0       14597
0.1      3557
0.15     2930
0.05     2768
0.2      1079
0.25      845
Name: count, dtype: int64

In [20]:
df_seduh_order['discount_pct'] = pd.to_numeric(
                    df_seduh_order['discount_pct']
                    .str.strip(),
                    errors='coerce'
)

In [21]:
print(df_seduh_order['discount_pct'].describe())
print(df_seduh_order['discount_pct'].isna().sum())
print(df_seduh_order['discount_pct'].sample(5))

count    25776.000000
mean         0.052787
std          0.071715
min          0.000000
25%          0.000000
50%          0.000000
75%          0.100000
max          0.250000
Name: discount_pct, dtype: float64
0
12261    0.0
2171     0.1
14596    0.0
20368    0.0
949      0.2
Name: discount_pct, dtype: float64


### 1.7 `channel` — seragamkan penulisan

In [22]:
df_seduh_order['channel'].value_counts(dropna=False)

channel
Tokopedia       7308
Shopee          6953
Webstore        5414
TikTok Shop     5342
Webstore          55
 Webstore         52
tokopedia         51
WEBSTORE          48
 Shopee           47
Tiktok shop       46
Shoppee           45
TOKOPEDIA         40
TIKTOK SHOP       39
Shopee            39
Tiktok Shop       37
Tokopedia         36
Tokopdia          35
webstore          35
tiktok shop       34
TikTok shop       34
shopee            30
SHOPEE            29
 Tokopedia        27
Name: count, dtype: int64

In [23]:
key=(
    df_seduh_order['channel']
    .str.strip()
    .str.lower()
)
ambil_channel=(
    df_seduh_order['channel']
    .str.strip()
    .str.lower()
    .drop_duplicates()
)

In [24]:
mapping_channel=dict.fromkeys(ambil_channel, None)
mapping_channel.update({
 'shopee': 'Shopee',
 'tiktok shop': 'TikTok Shop',
 'webstore': 'Webstore',
 'tokopedia': 'Tokopedia',
 'shoppee': 'Shopee',
 'tokopdia': 'Tokopedia'
})

In [25]:
df_seduh_order['channel'] = key.map(mapping_channel)


In [26]:
print(df_seduh_order['channel'].value_counts(dropna=False))
print(df_seduh_order['channel'].isna().sum())

channel
Tokopedia      7497
Shopee         7143
Webstore       5604
TikTok Shop    5532
Name: count, dtype: int64
0


### 1.8 `payment_method`

In [27]:
df_seduh_order['payment_method'].value_counts(dropna=False)

payment_method
Bank Transfer    4192
GoPay            4153
ShopeePay        3813
DANA             3156
COD              3059
OVO              2860
Paylater         2321
Credit Card      2222
Name: count, dtype: int64

### 1.9 `city` & `province`

In [28]:
df_seduh_order['city'].value_counts(dropna=False)

city
Balikpapan        4710
Bandung           2793
Surabaya          2437
Semarang          1524
Bekasi            1485
                  ... 
Palembang            3
DENPASAR             2
PEKANBARU            2
Depok                2
BANDAR LAMPUNG       2
Name: count, Length: 88, dtype: int64

In [29]:
key2=(
    df_seduh_order['city']
    .str.strip()
    .str.lower()
)
ambil_kota=(
    df_seduh_order['city']
    .str.strip()
    .str.lower()
    .drop_duplicates()
)

In [30]:
mapping_kota=dict.fromkeys(ambil_kota, None)
mapping_kota.update({
 'balikpapan': 'Balikpapan',
 'palembang': 'Palembang',
 'tangerang': 'Tangerang',
 'semarang': 'Semarang',
 'depok': 'Depok',
 'bekasi': 'Bekasi',
 'bandung': 'Bandung',
 'bogor': 'Bogor',
 'malang': 'Malang',
 'solo': 'Solo',
 'surabaya': 'Surabaya',
 'denpasar': 'Denpasar',
 'pekanbaru': 'Pekanbaru',
 'makassar': 'Makassar',
 'bandar lampung': 'Bandar Lampung',
 'medan': 'Medan',
 'yogyakarta': 'Yogyakarta',
 'jakarta': 'Jakarta'
})

In [31]:
df_seduh_order['city']=key2.map(mapping_kota)

print(key2.loc[df_seduh_order['city'].isna()])
print(key2.loc[df_seduh_order['city'].isna()].unique())

Series([], Name: city, dtype: str)
<StringArray>
[]
Length: 0, dtype: str


In [32]:
print(df_seduh_order['province'].value_counts(dropna=False))
print(df_seduh_order['province'].isna().sum())

province
Jawa Barat          6569
Kalimantan Timur    4820
Jawa Timur          3695
Jawa Tengah         2280
Sumatera Utara      1313
Banten              1279
DI Yogyakarta       1022
DKI Jakarta         1002
Bali                 966
Sumatera Selatan     796
Lampung              751
Sulawesi Selatan     713
Riau                 570
Name: count, dtype: int64
0


### 1.10 `order_status`, `rating`, `review_text`

In [33]:
print(df_seduh_order['order_status'].value_counts(dropna=False))
print(df_seduh_order['order_status'].isna().sum())

order_status
Completed    23971
Returned      1038
Cancelled      767
Name: count, dtype: int64
0


In [34]:
print(df_seduh_order['rating'].value_counts(dropna=False))
print(df_seduh_order['rating'].isna().sum())

rating
5      10289
4       8267
NaN     4077
3       2595
2        447
1        101
Name: count, dtype: int64
4077


In [35]:
df_seduh_order['rating'] = pd.to_numeric(
                df_seduh_order['rating']
                .str.strip(),
                errors='coerce'
)

In [36]:
df_seduh_order['rating'] = (df_seduh_order['rating'].astype('Int64'))

In [37]:
print(df_seduh_order['rating'].describe())
print(df_seduh_order['rating'].isna().sum())
print(df_seduh_order['rating'].value_counts(dropna=False))
print(df_seduh_order['rating'].isna().sum())
print(df_seduh_order['rating'].info())

count     21699.0
mean     4.299415
std      0.792737
min           1.0
25%           4.0
50%           4.0
75%           5.0
max           5.0
Name: rating, dtype: Float64
4077
rating
5       10289
4        8267
<NA>     4077
3        2595
2         447
1         101
Name: count, dtype: Int64
4077
<class 'pandas.Series'>
Index: 25776 entries, 0 to 25803
Series name: rating
Non-Null Count  Dtype
--------------  -----
21699 non-null  Int64
dtypes: Int64(1)
memory usage: 427.9 KB
None


In [38]:
print(df_seduh_order['review_text'].value_counts(dropna=False).head(20))
print(df_seduh_order['review_text'].isna().sum())

review_text
NaN                                                          16686
Langganan bulanan praktis, tidak perlu repot pesan ulang.      776
Pengiriman cepat, kemasan aman.                                762
Kopinya wangi, roast-nya pas untuk V60.                        758
Worth it untuk harganya, akan repeat order.                    745
Packaging rapi, biji utuh dan segar.                           734
Cocok buat manual brew harian.                                 731
Sudah order berkali-kali, kualitas konsisten.                  731
Rasa manis alami, nggak pahit berlebihan.                      728
Gift set-nya cantik, penerima suka banget.                     722
Aroma dan rasanya mantap, fresh banget.                        711
Standar, tidak istimewa tapi tidak mengecewakan.               215
Kemasan sedikit penyok tapi isi aman.                          205
Enak tapi agak asam untuk selera saya.                         196
Sesuai deskripsi, harga agak mahal.               

In [39]:
df_seduh_order['review_text'] = (
         df_seduh_order['review_text']
         .astype('string')
         .str.strip()
         .replace('', pd.NA)
)

### 1.11 Cek akhir

In [40]:
df_seduh_order['order_date'].sample(5)

25120   2025-12-16
10278   2024-11-25
12690   2025-01-31
16372   2025-05-18
1273    2024-02-17
Name: order_date, dtype: datetime64[us]

In [41]:
df_seduh_order.info()

<class 'pandas.DataFrame'>
Index: 25776 entries, 0 to 25803
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        25776 non-null  str           
 1   order_date      25776 non-null  datetime64[us]
 2   customer_id     25776 non-null  str           
 3   product_id      25776 non-null  str           
 4   quantity        25776 non-null  int64         
 5   unit_price      25776 non-null  int64         
 6   discount_pct    25776 non-null  float64       
 7   channel         25776 non-null  str           
 8   payment_method  25776 non-null  str           
 9   city            25776 non-null  str           
 10  province        25776 non-null  str           
 11  order_status    25776 non-null  str           
 12  rating          21699 non-null  Int64         
 13  review_text     9090 non-null   string        
dtypes: Int64(1), datetime64[us](1), float64(1), int64(2), str(8), string(1

## 2. Customers

In [ ]:
df_seduh_cust_raw = pd.read_excel(FILE_PATH, sheet_name='Customers', dtype=str)
df_seduh_cust = df_seduh_cust_raw.copy()

### 2.1 Cek struktur awal & `customer_id`

In [42]:
print(df_seduh_cust.shape)
print(df_seduh_cust['customer_id'].duplicated().sum())
print(df_seduh_cust['customer_id'].isna().sum())
print(df_seduh_cust.info())

(15000, 9)
0
0
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   customer_id          15000 non-null  str  
 1   customer_name        15000 non-null  str  
 2   phone                15000 non-null  str  
 3   gender               15000 non-null  str  
 4   age                  14790 non-null  str  
 5   city                 14620 non-null  str  
 6   province             15000 non-null  str  
 7   signup_date          15000 non-null  str  
 8   acquisition_channel  15000 non-null  str  
dtypes: str(9)
memory usage: 1.0 MB
None


In [43]:
df_seduh_cust.loc[df_seduh_cust['customer_id'].str.len() > 10]

,customer_id,customer_name,phone,gender,age,city,province,signup_date,acquisition_channel


### 2.2 `customer_name` — rapikan spasi & kapitalisasi

In [44]:
print(df_seduh_cust['customer_name'].isna().sum())
print(df_seduh_cust['customer_name'].duplicated().sum())
print(df_seduh_cust['customer_name'].value_counts(dropna=False))

0
12525
customer_name
Ivan Prakoso        18
Ulfa Permana        18
Cindy Lestari       17
Zahra Maulana       17
Ayu Pratama         17
                    ..
 Adit Kurniawan      1
TONO LESTARI         1
Doni  Cahyani        1
 Endah Anggraini     1
IVAN SAPUTRA         1
Name: count, Length: 2475, dtype: int64


In [45]:
cust_name=(
    df_seduh_cust['customer_name']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

In [46]:
cust_name.sample(5)

10725    Jasmine Kurniawan
4980             Oki Halim
6697         Dinda Santoso
10432        Kevin Maulana
13949         Cindy Wibowo
Name: customer_name, dtype: str

In [47]:
df_seduh_cust['customer_name'] = cust_name
print(df_seduh_cust['customer_name'].isna().sum())
print(df_seduh_cust['customer_name'].duplicated().sum())
print(df_seduh_cust['customer_name'].value_counts(dropna=False))

0
13200
customer_name
Ivan Prakoso     18
Irfan Hidayat    18
Ulfa Permana     18
Erik Utami       17
Cindy Lestari    17
                 ..
Beni Prakoso      1
Umar Handoko      1
Zahra Cahyani     1
Eka Nugroho       1
Laras Wibowo      1
Name: count, Length: 1800, dtype: int64


### 2.3 `phone` — seragamkan jadi format `08xxxxxxxxxx`

In [48]:
print(df_seduh_cust['phone'].value_counts(dropna=False).head(20))
print(df_seduh_cust['phone'].isna().sum())

phone
0891-4744-2515    1
081187744944      1
+6285239805833    1
0814-6016-7147    1
+6288792942660    1
0881-8914-1808    1
+6282936917676    1
0871-5049-2604    1
0890-8284-6189    1
089435261945      1
085896467726      1
+6288957876031    1
+6289324439364    1
0897-9886-8656    1
+6284716299620    1
083723904244      1
+6286019708879    1
081855134414      1
0882-5454-5775    1
0887-6405-9923    1
Name: count, dtype: int64
0


In [49]:
phone=(
    df_seduh_cust['phone']
    .str.strip()
    .str.replace(r'[^\d]', '', regex=True)
    .str.replace(r'^62', '0', regex=True)
)

In [50]:
phone.loc[phone.str.fullmatch(r'^62\d{10}')]

Series([], Name: phone, dtype: str)

In [51]:
print(phone.str.fullmatch(r'08\d{10}').value_counts(dropna=False))

phone
True    15000
Name: count, dtype: int64


In [52]:
df_seduh_cust['phone'] = phone
print(df_seduh_cust['phone'].str.fullmatch(r'08\d{10}').value_counts(dropna=False))

phone
True    15000
Name: count, dtype: int64


### 2.4 `age` → numerik

In [53]:
df_seduh_cust['age'].map(type).value_counts(dropna=False)

age
<class 'str'>      14790
<class 'float'>      210
Name: count, dtype: int64

In [54]:
df_seduh_cust.loc[~df_seduh_cust['age'].str.fullmatch(r'\d+', na=False)]

,customer_id,customer_name,phone,gender,age,city,province,signup_date,acquisition_channel
74,CUST-00075,Omar Cahyani,087254606356,Male,NaN,Semarang,Jawa Tengah,2022-02-24 00:00:00,Instagram Ads
174,CUST-00175,Bayu Darmawan,082415758594,Male,NaN,Tangerang,Banten,2022-01-27 00:00:00,Organic Social
403,CUST-00404,Jihan Anggraini,086424891190,Male,NaN,Surabaya,Jawa Timur,2022-05-19 00:00:00,Instagram Ads
422,CUST-00423,Krisna Pratama,089435014547,Male,NaN,Tangerang,Banten,2022-11-23 00:00:00,Referral
632,CUST-00633,Umar Cahyani,089870103194,Male,NaN,Balikpapan,Kalimantan Timur,2022-12-28 00:00:00,TikTok Ads
...,...,...,...,...,...,...,...,...,...
14707,CUST-14708,Nadia Firmansyah,083579829113,MALE,NaN,Medan,Sumatera Utara,2025-01-17 00:00:00,TikTok Ads
14734,CUST-14735,Endah Kurniawan,085937844632,Female,NaN,Balikpapan,Kalimantan Timur,2025-03-18 00:00:00,Marketplace
14742,CUST-14743,Sinta Anggraini,082512417458,Female,NaN,Jakarta,DKI Jakarta,2025-02-21 00:00:00,TikTok Ads
14831,CUST-14832,Faisal Pratama,089532457987,Female,NaN,Surabaya,Jawa Timur,2025-02-01 00:00:00,Organic Social


In [55]:
df_seduh_cust['age'].info()

<class 'pandas.Series'>
RangeIndex: 15000 entries, 0 to 14999
Series name: age
Non-Null Count  Dtype
--------------  -----
14790 non-null  str  
dtypes: str(1)
memory usage: 117.3 KB


In [56]:
age=pd.to_numeric(
    df_seduh_cust['age']
    .str.strip(),
    errors='coerce'
).astype('Int64')

In [57]:
df_seduh_cust['age']=age

In [58]:
df_seduh_cust['age'].describe()

count      14790.0
mean     29.142799
std       6.090647
min           19.0
25%           25.0
50%           29.0
75%           33.0
max           45.0
Name: age, dtype: Float64

### 2.5 `gender` — seragamkan

In [59]:
key_gender=(
    df_seduh_cust['gender']
    .str.strip()
    .str.lower()
)
ambil_gender=(
    df_seduh_cust['gender']
    .str.strip()
    .str.lower()
    .drop_duplicates()
)

In [60]:
mapping_gender=dict.fromkeys(ambil_gender, None)
mapping_gender.update({
 'male': 'Male',
 'female': 'Female',
 'm': 'Male',
 'f': 'Female',
 'perempuan': 'Female',
 'laki-laki': 'Male'
})

In [61]:
gender=key_gender.map(mapping_gender)
df_seduh_cust['gender']=gender

### 2.6 `city`, `province`, `acquisition_channel` — mapping penulisan baku

In [62]:
key_kota=(
    df_seduh_cust['city']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)
key_prov=(
    df_seduh_cust['province']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)
key_ac_chan=(
    df_seduh_cust['acquisition_channel']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)
ambil_kota=key_kota.dropna().unique()
ambil_prov=key_prov.dropna().unique()
ambil_ac_chan=key_ac_chan.dropna().unique()


In [63]:
mapping_kota=dict.fromkeys(ambil_kota, None)
mapping_prov=dict.fromkeys(ambil_prov, None)
mapping_ac_chan=dict.fromkeys(ambil_ac_chan, None)
mapping_kota.update({
 'yogyakarta': 'Yogyakarta',
 'bandung': 'Bandung',
 'semarang': 'Semarang',
 'tangerang': 'Tangerang',
 'depok': 'Depok',
 'jakarta': 'Jakarta',
 'balikpapan': 'Balikpapan',
 'palembang': 'Palembang',
 'bekasi': 'Bekasi',
 'surabaya': 'Surabaya',
 'denpasar': 'Denpasar',
 'solo': 'Solo',
 'medan': 'Medan',
 'malang': 'Malang',
 'bogor': 'Bogor',
 'makassar': 'Makassar',
 'pekanbaru': 'Pekanbaru',
 'bandar lampung': 'Bandar Lampung'
})
mapping_prov.update({
 'di yogyakarta': 'DI Yogyakarta',
 'jawa barat': 'Jawa Barat',
 'jawa tengah': 'Jawa Tengah',
 'banten': 'Banten',
 'dki jakarta': 'DKI Jakarta',
 'kalimantan timur': 'Kalimantan Timur',
 'sumatera selatan': 'Sumatera Selatan',
 'jawa timur': 'Jawa Timur',
 'bali': 'Bali',
 'sumatera utara': 'Sumatera Utara',
 'sulawesi selatan': 'Sulawesi Selatan',
 'riau': 'Riau',
 'lampung': 'Lampung'
})
mapping_ac_chan.update({
 'google search': 'Google Search',
 'organic social': 'Organic Social',
 'marketplace': 'Marketplace',
 'tiktok ads': 'TikTok Ads',
 'instagram ads': 'Instagram Ads',
 'referral': 'Referral'
})

In [64]:
df_seduh_cust['city']=key_kota.map(mapping_kota)
df_seduh_cust['province']=key_prov.map(mapping_prov)
df_seduh_cust['acquisition_channel']=key_ac_chan.map(mapping_ac_chan)

In [ ]:
df_seduh_cust['city']=df_seduh_cust['city'].fillna('Unknown')
print(df_seduh_cust['city'].isna().sum())            
print((df_seduh_cust['city']=='Unknown').sum())     

0
380


In [65]:
df_seduh_cust.columns

Index(['customer_id', 'customer_name', 'phone', 'gender', 'age', 'city',
       'province', 'signup_date', 'acquisition_channel'],
      dtype='str')

### 2.7 `signup_date` → datetime

In [66]:
df_seduh_cust['signup_date'].map(type).value_counts(dropna=False)

signup_date
<class 'str'>    15000
Name: count, dtype: int64

In [67]:
df_seduh_cust.loc[
    ~df_seduh_cust['signup_date'].str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False)
    & (df_seduh_cust['signup_date'].str.len() == 10)
]

,customer_id,customer_name,phone,gender,age,city,province,signup_date,acquisition_channel


In [68]:
df_seduh_cust['signup_date']=pd.to_datetime(
                                df_seduh_cust['signup_date'],
                                errors='coerce'
)

In [69]:
df_seduh_cust.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   customer_id          15000 non-null  str           
 1   customer_name        15000 non-null  str           
 2   phone                15000 non-null  str           
 3   gender               15000 non-null  str           
 4   age                  14790 non-null  Int64         
 5   city                 15000 non-null  str           
 6   province             15000 non-null  str           
 7   signup_date          15000 non-null  datetime64[us]
 8   acquisition_channel  15000 non-null  str           
dtypes: Int64(1), datetime64[us](1), str(7)
memory usage: 1.0 MB


### 2.8 Cek akhir

In [ ]:

print(df_seduh_cust['acquisition_channel'].value_counts())

acquisition_channel
Instagram Ads     3647
Organic Social    2600
TikTok Ads        2428
Google Search     2398
Marketplace       1983
Referral          1944
Name: count, dtype: int64


## 3. Products

In [ ]:
df_seduh_prod_raw = pd.read_excel(FILE_PATH, sheet_name='Products', dtype=str)
df_seduh_prod = df_seduh_prod_raw.copy()

### 3.1 Cek struktur, `product_id`, `sku_code`, `product_name`

In [70]:
df_seduh_prod.info()
print(df_seduh_prod['product_id'].isna().sum())
print(df_seduh_prod['product_id'].duplicated().sum())
print(df_seduh_prod['product_id'].map(type).value_counts())

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    50 non-null     str  
 1   sku_code      50 non-null     str  
 2   product_name  50 non-null     str  
 3   category      50 non-null     str  
 4   base_price    50 non-null     str  
 5   unit_cost     50 non-null     str  
 6   launch_date   50 non-null     str  
dtypes: str(7)
memory usage: 2.9 KB
0
0
product_id
<class 'str'>    50
Name: count, dtype: int64


In [71]:
print(df_seduh_prod['sku_code'].isna().sum())
print(df_seduh_prod['sku_code'].duplicated().sum())
print(df_seduh_prod['sku_code'].map(type).value_counts())
print(df_seduh_prod['sku_code'].map(repr).value_counts())


0
0
sku_code
<class 'str'>    50
Name: count, dtype: int64
sku_code
'RB-ARABIKAG-001'     1
'RB-ARABIKAT-002'     1
'RB-ARABIKAK-003'     1
'RB-ARABIKAJ-004'     1
'RB-ARABIKAF-005'     1
'RB-ROBUSTAL-006'     1
'RB-HOUSEBLE-007'     1
'RB-HOUSEBLE-008'     1
'RB-HOUSEBLE-009'     1
'RB-ESPRESSO-010'     1
'RB-DECAFBLE-011'     1
'RB-SINGLEOR-012'     1
'RB-MANUKGAD-013'     1
'RB-COLDBREW-014'     1
'RTD-COLDBREW-015'    1
'RTD-COLDBREW-016'    1
'RTD-ESKOPI-017'      1
'RTD-AMERICAN-018'    1
'RTD-LATTECAN-019'    1
'RTD-RTDVARIE-020'    1
'RTD-NITROCOL-021'    1
'BE-DRIPPERC-022'     1
'BE-FRENCHPR-023'     1
'BE-AEROPRES-024'     1
'BE-MOKAPOT-025'      1
'BE-MANUALHA-026'     1
'BE-GOOSENEC-027'     1
'BE-DIGITALC-028'     1
'BE-ESPRESSO-029'     1
'BE-ELECTRIC-030'     1
'BE-CHEMEX-031'       1
'AC-PAPERFIL-032'     1
'AC-REUSABLE-033'     1
'AC-CLEANING-034'     1
'AC-AIRTIGHT-035'     1
'AC-MILKFROT-036'     1
'AC-COFFEEKN-037'     1
'AC-INSULATE-038'     1
'AC-FILTERPA-039'   

In [72]:
print(df_seduh_prod['product_name'].isna().sum())
print(df_seduh_prod['product_name'].duplicated().sum())
print(df_seduh_prod['product_name'].map(type).value_counts())
print(df_seduh_prod['product_name'].map(repr).value_counts())

0
0
product_name
<class 'str'>    50
Name: count, dtype: int64
product_name
'Arabika Gayo Aceh 200g'             1
'Arabika Toraja Sapan 200g'          1
'Arabika Kintamani Bali 200g'        1
'Arabika Java Preanger 200g'         1
'Arabika Flores Bajawa 200g'         1
'Robusta Lampung 250g'               1
'House Blend 200g'                   1
'House Blend 500g'                   1
'House Blend 1kg'                    1
'Espresso Blend 250g'                1
'Decaf Blend 200g'                   1
'Single Origin Sampler 4x100g'       1
'Manuk Gading Natural 200g'          1
'Cold Brew Blend 500g'               1
'Cold Brew Original 250ml'           1
'Cold Brew Latte 250ml'              1
'Es Kopi Susu Gula Aren 250ml'       1
'Americano Can 200ml'                1
'Latte Can 200ml'                    1
'RTD Variety Pack 6x250ml'           1
'Nitro Cold Brew 250ml'              1
'V60 Dripper Ceramic'                1
'French Press 600ml'                 1
'Aeropress Coffee Maker'   

### 3.2 `category` — seragamkan penulisan

In [73]:
print(df_seduh_prod['category'].isna().sum())
print(df_seduh_prod['category'].duplicated().sum())
print(df_seduh_prod['category'].map(type).value_counts())
print(df_seduh_prod['category'].map(repr).value_counts())

0
41
category
<class 'str'>    50
Name: count, dtype: int64
category
'Roasted Beans'        13
'Brewing Equipment'    10
'Accessories'           8
'Ready-to-Drink'        5
'Subscription'          5
'Gift Set'              5
'Ready to Drink'        2
'Roasted Beans '        1
'Gift set'              1
Name: count, dtype: int64


In [74]:
key_cat=(
    df_seduh_prod['category']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)
ambil_cat=key_cat.dropna().unique()

In [75]:
mapping_cat=dict.fromkeys(ambil_cat, None)
mapping_cat.update({
 'roasted beans': 'Roasted Beans',
 'ready-to-drink': 'Ready to Drink',
 'ready to drink': 'Ready to Drink',
 'brewing equipment': 'Brewing Equipment',
 'accessories': 'Accessories',
 'subscription': 'Subscription',
 'gift set': 'Gift Set'
})

In [76]:
df_seduh_prod['category']=key_cat.map(mapping_cat)

In [77]:
df_seduh_prod['category'].map(repr).value_counts()

category
'Roasted Beans'        14
'Brewing Equipment'    10
'Accessories'           8
'Ready to Drink'        7
'Gift Set'              6
'Subscription'          5
Name: count, dtype: int64

### 3.3 `base_price` & `unit_cost` → angka

In [78]:
print(df_seduh_prod['base_price'].isna().sum())
print(df_seduh_prod['base_price'].duplicated().sum())
print(df_seduh_prod['base_price'].map(type).value_counts())
print(df_seduh_prod['base_price'].map(repr).value_counts())

0
7
base_price
<class 'str'>    50
Name: count, dtype: int64
base_price
'165000'     3
'145000'     3
'125000'     2
'89000'      2
'135000'     2
'115000'     1
'118000'     1
'110000'     1
'122000'     1
'78000'      1
'195000'     1
'349000'     1
'99000'      1
'178000'     1
'32000'      1
'38000'      1
'28000'      1
'25000'      1
'30000'      1
'42000'      1
'235000'     1
'425000'     1
'285000'     1
'389000'     1
'315000'     1
'275000'     1
'950000'     1
'520000'     1
'45000'      1
'35000'      1
'95000'      1
'65000'      1
'149000'     1
'259000'     1
'289000'     1
'339000'     1
'219000'     1
'385000'     1
'295000'     1
'675000'     1
'245000'     1
'525000'     1
'1250000'    1
Name: count, dtype: int64


In [79]:
df_seduh_prod.loc[~df_seduh_prod['base_price'].str.fullmatch(r'\d+')]

,product_id,sku_code,product_name,category,base_price,unit_cost,launch_date


In [80]:
df_seduh_prod.loc[~df_seduh_prod['unit_cost'].str.fullmatch(r'\d+')]

,product_id,sku_code,product_name,category,base_price,unit_cost,launch_date


In [81]:
df_seduh_prod['base_price']=pd.to_numeric(
                              df_seduh_prod['base_price']
                              .str.strip()
                              .str.extract(r'(\d+)')[0],
                              errors='coerce'  
).astype('Int64')

df_seduh_prod['unit_cost']=pd.to_numeric(
                              df_seduh_prod['unit_cost']
                              .str.strip()
                              .str.extract(r'(\d+)')[0],
                              errors='coerce'  
).astype('Int64')

In [82]:
print(df_seduh_prod[['base_price', 'unit_cost']].isna().sum())
print((df_seduh_prod['unit_cost'] >= df_seduh_prod['base_price']).sum())
print(df_seduh_prod[['base_price', 'unit_cost']].describe())

base_price    0
unit_cost     0
dtype: int64
0
          base_price      unit_cost
count           50.0           50.0
mean        229020.0       126540.0
std    231728.596528  144860.553637
min          25000.0        12000.0
25%          96000.0        45875.0
50%         147000.0        74000.0
75%         288000.0       144500.0
max        1250000.0       700000.0


### 3.4 `launch_date` → datetime

In [83]:
df_seduh_prod['launch_date'] = pd.to_datetime(
                            df_seduh_prod['launch_date']
                            .str.strip(),
                            errors='coerce'
)

In [84]:
print(df_seduh_prod['launch_date'].isna().sum())

0


In [85]:
print(df_seduh_prod['launch_date'].min(), df_seduh_prod['launch_date'].max())

2022-01-07 00:00:00 2024-06-09 00:00:00


### 3.5 Cek akhir

In [86]:
df_seduh_prod.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   product_id    50 non-null     str           
 1   sku_code      50 non-null     str           
 2   product_name  50 non-null     str           
 3   category      50 non-null     str           
 4   base_price    50 non-null     Int64         
 5   unit_cost     50 non-null     Int64         
 6   launch_date   50 non-null     datetime64[us]
dtypes: Int64(2), datetime64[us](1), str(4)
memory usage: 3.0 KB


## 4. Marketing_Spend

In [ ]:
df_seduh_MS_raw = pd.read_excel(FILE_PATH, sheet_name='Marketing_Spend', dtype=str)
df_seduh_MS = df_seduh_MS_raw.copy()

### 4.1 Cek struktur awal

In [87]:
df_seduh_MS.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   month        96 non-null     str  
 1   channel      96 non-null     str  
 2   spend_idr    96 non-null     str  
 3   impressions  96 non-null     str  
 4   clicks       96 non-null     str  
dtypes: str(5)
memory usage: 3.9 KB


In [88]:
df_seduh_MS.columns

Index(['month', 'channel', 'spend_idr', 'impressions', 'clicks'], dtype='str')

### 4.2 `month` — cek format & duplikat, lalu ubah ke datetime

In [89]:
print(df_seduh_MS['month'].isna().sum())
print(df_seduh_MS['month'].duplicated().sum())
print(df_seduh_MS['month'].map(type).value_counts())
print(df_seduh_MS['month'].map(repr).value_counts())

0
72
month
<class 'str'>    96
Name: count, dtype: int64
month
'2024-01-01 00:00:00'    4
'2024-02-01 00:00:00'    4
'2024-03-01 00:00:00'    4
'2024-04-01 00:00:00'    4
'2024-05-01 00:00:00'    4
'2024-06-01 00:00:00'    4
'2024-07-01 00:00:00'    4
'2024-08-01 00:00:00'    4
'2024-09-01 00:00:00'    4
'2024-10-01 00:00:00'    4
'2024-11-01 00:00:00'    4
'2024-12-01 00:00:00'    4
'2025-01-01 00:00:00'    4
'2025-02-01 00:00:00'    4
'2025-03-01 00:00:00'    4
'2025-04-01 00:00:00'    4
'2025-05-01 00:00:00'    4
'2025-06-01 00:00:00'    4
'2025-07-01 00:00:00'    4
'2025-08-01 00:00:00'    4
'2025-09-01 00:00:00'    4
'2025-10-01 00:00:00'    4
'2025-11-01 00:00:00'    4
'2025-12-01 00:00:00'    4
Name: count, dtype: int64


In [90]:
df_seduh_MS.loc[~df_seduh_MS['month'].str.fullmatch(r'\d{4}-\d{2}-\d{2}')]

,month,channel,spend_idr,impressions,clicks
0,2024-01-01 00:00:00,Google Search,3160000,115240,1743
1,2024-01-01 00:00:00,Instagram Ads,9220000,442423,4603
2,2024-01-01 00:00:00,Marketplace Ads,4170000,188043,2572
3,2024-01-01 00:00:00,TikTok Ads,6820000,252310,2334
4,2024-02-01 00:00:00,Google Search,3420000,184208,1698
...,...,...,...,...,...
91,2025-11-01 00:00:00,TikTok Ads,20020000,1086933,15346
92,2025-12-01 00:00:00,Google Search,9850000,387591,5478
93,2025-12-01 00:00:00,Instagram Ads,25560000,912361,11828
94,2025-12-01 00:00:00,Marketplace Ads,12080000,553167,8336


In [91]:
df_seduh_MS.duplicated(['month', 'channel']).sum()

np.int64(0)

In [92]:
df_seduh_MS['month']=pd.to_datetime(
                df_seduh_MS['month']
                .str.strip(),
                errors='coerce'
)

In [93]:
print(df_seduh_MS['month'].isna().sum())
print(df_seduh_MS['month'].min(), df_seduh_MS['month'].max())

0
2024-01-01 00:00:00 2025-12-01 00:00:00


### 4.3 Cek kelengkapan data per bulan & ringkasan angka

In [94]:
df_seduh_MS.groupby('month').size().value_counts()

4    24
Name: count, dtype: int64

In [95]:
df_seduh_MS[['spend_idr', 'impressions', 'clicks']].describe()

,spend_idr,impressions,clicks
count,96,96,96
unique,94,96,95
top,5220000,115240,5743
freq,2,1,2


### 4.4 Cek akhir

In [96]:
df_seduh_MS.info()


<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   month        96 non-null     datetime64[us]
 1   channel      96 non-null     str           
 2   spend_idr    96 non-null     str           
 3   impressions  96 non-null     str           
 4   clicks       96 non-null     str           
dtypes: datetime64[us](1), str(4)
memory usage: 3.9 KB


## 5. Export

In [ ]:
df_seduh_order.to_csv('Orders.csv', index=False)
df_seduh_cust.to_csv('Customers.csv', index=False)
df_seduh_prod.to_csv('Products.csv', index=False)
df_seduh_MS.to_csv('Marketing_Spend.csv', index=False)